# 🔧 Data Preprocessing and Augmentation

## Document Type Classification - Part 2

**Goal**: Prepare the data pipeline for model training

In this notebook, we will:
1. Build image loading and preprocessing functions
2. Implement data augmentation techniques
3. Create data generators for efficient training
4. Validate the pipeline before training

**Why is this important?**
- Raw images have different sizes → need standardization
- Limited data (1,470 train images) → augmentation increases diversity
- Proper preprocessing prevents overfitting and improves generalization

**Key Insight from Notebook 01:**
- Passports: 1136×803 pixels
- ID Cards: 1519×1069 pixels
- Licenses: 1013×638 pixels
→ We need to standardize all to 224×224


## 1. Setup and Imports

We'll use **TensorFlow/Keras** for data preprocessing and augmentation:
- `ImageDataGenerator`: Built-in augmentation capabilities
- `tf.keras.utils`: Image preprocessing utilities
- Efficient batch loading during training


In [ ]:
# Suppress TensorFlow warnings and optimize loading
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Reduce TensorFlow logging
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'  # Disable oneDNN optimizations for faster startup

# Data handling
import pandas as pd
import numpy as np
from pathlib import Path

# Image processing
from PIL import Image

# TensorFlow/Keras
print('⏳ Loading TensorFlow (this may take 5-10 seconds)...')
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

print('✅ All libraries imported successfully!')
print(f'   TensorFlow version: {tf.__version__}')
print(f'   Running on: {"GPU" if tf.config.list_physical_devices("GPU") else "CPU"}')


## 2. Image Preprocessing

**Why do we need preprocessing?**

From notebook 01, we discovered that our images have different dimensions:
- Passports: 1136×803 pixels (aspect ratio 1.41)
- ID Cards: 1519×1069 pixels (aspect ratio 1.42)
- Licenses: 1013×638 pixels (aspect ratio 1.59)

**Problems:**
1. **Different sizes** → CNNs expect fixed input dimensions
2. **Large images** → Slow training and high memory usage
3. **Pixel values 0-255** → Need normalization for better convergence

**Our Solution:**
1. **Resize** all images to 224×224 (standard for pre-trained models like ResNet)
2. **Normalize** pixel values to [0, 1] range
3. **Convert** to numpy arrays for TensorFlow

**Note**: Resizing will slightly distort aspect ratios, but this is acceptable and common practice in computer vision.


In [ ]:
# Load datasets
DATA_DIR = '../../datasets/idnet/document_type_classification/data/'
train_df = pd.read_csv(DATA_DIR + 'train_dataset.csv')
val_df = pd.read_csv(DATA_DIR + 'val_dataset.csv')
test_df = pd.read_csv(DATA_DIR + 'test_dataset.csv')

print(f'📂 Datasets loaded:')
print(f'   Train: {len(train_df):,} images')
print(f'   Validation: {len(val_df):,} images')
print(f'   Test: {len(test_df):,} images')

# Test loading one image
sample_path = train_df.iloc[0]['image_path']
sample_img = Image.open(sample_path)

print(f'\n🖼️  Sample image (before preprocessing):')
print(f'   Path: {os.path.basename(sample_path)}')
print(f'   Size: {sample_img.size} (width × height)')
print(f'   Mode: {sample_img.mode}')

# Display original image
plt.figure(figsize=(6, 6))
plt.imshow(sample_img)
plt.title('Original Image (Before Preprocessing)', fontsize=12, fontweight='bold')
plt.axis('off')
plt.show()


## 2.1. Preprocessing Function

Now we'll create a function that:
1. Loads an image from a file path
2. Resizes it to 224×224 pixels
3. Converts to numpy array
4. Normalizes pixel values to [0, 1]

**Why 224×224?**
- Standard size for many pre-trained models (ResNet, VGG, EfficientNet)
- Good balance between detail and computational efficiency
- Allows us to use transfer learning later


In [ ]:
def load_and_preprocess_image(path, target_size=(224, 224)):
    """
    Load and preprocess a single image.
    
    Args:
        path (str): Path to image file
        target_size (tuple): Target dimensions (width, height)
    
    Returns:
        numpy.ndarray: Preprocessed image array with shape (224, 224, 3) and values in [0, 1]
    """
    # Load image
    img = Image.open(path)
    
    # Resize to target size
    img = img.resize(target_size)
    
    # Convert to numpy array
    img_array = np.array(img)
    
    # Normalize to [0, 1]
    img_array = img_array / 255.0
    
    return img_array

# Test the function
preprocessed_img = load_and_preprocess_image(sample_path)

print('✅ Preprocessing function created!')
print(f'\n📊 Image transformation:')
print(f'   Original size: {sample_img.size}')
print(f'   After resize: {preprocessed_img.shape}')
print(f'   Value range: [{preprocessed_img.min():.3f}, {preprocessed_img.max():.3f}]')
print(f'   Data type: {preprocessed_img.dtype}')

# Display preprocessed image
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(sample_img)
axes[0].set_title(f'Original ({sample_img.size[0]}×{sample_img.size[1]})', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(preprocessed_img)
axes[1].set_title('Preprocessed (224×224)', fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()


## 3. Data Augmentation

**What is Data Augmentation?**

Data augmentation is the technique of creating modified versions of training images to artificially increase dataset size and diversity.

**Why do we need it?**

Our training set has only **1,470 images**. This is relatively small for training a deep neural network, which can lead to:
- **Overfitting**: Model memorizes training data instead of learning general patterns
- **Poor generalization**: Model fails on new, unseen documents

**How does augmentation help?**

By applying random transformations, we can create thousands of variations:
- 1,470 original images × augmentation → effectively 10,000+ training samples
- Model learns to be invariant to lighting, rotation, position
- Better generalization to real-world conditions

**Augmentation techniques we'll use:**
1. **Rotation** (±15°): Documents might be slightly tilted when photographed
2. **Horizontal Flip**: Creates mirror images
3. **Brightness** (80-120%): Different lighting conditions
4. **Width/Height Shift** (±10%): Document not perfectly centered
5. **Zoom** (90-110%): Different camera distances

**What we WON'T use:**
- ❌ Vertical flip (documents are never upside down)
- ❌ Extreme rotations (>30°) (unrealistic)
- ❌ Color changes (document colors are important features)


## 3.1. Visualize Augmentation Effects

Let's see how augmentation transforms a single image. This helps us understand what the model will see during training.


In [ ]:
# Create augmentation generator
augmentation_demo = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    zoom_range=0.1,
    fill_mode='nearest'
)

# Prepare image for augmentation (need 4D array)
img_array = img_to_array(sample_img.resize((224, 224)))
img_array = img_array.reshape((1,) + img_array.shape)  # (1, 224, 224, 3)

# Generate 6 augmented versions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Data Augmentation Examples - Same Image, Different Transformations', 
             fontsize=14, fontweight='bold')

# Original
axes[0, 0].imshow(sample_img.resize((224, 224)))
axes[0, 0].set_title('Original', fontweight='bold')
axes[0, 0].axis('off')

# Generate 5 augmented versions
aug_iter = augmentation_demo.flow(img_array, batch_size=1)
positions = [(0,1), (0,2), (1,0), (1,1), (1,2)]

for idx, pos in enumerate(positions):
    augmented_img = next(aug_iter)[0].astype('uint8')
    axes[pos].imshow(augmented_img)
    axes[pos].set_title(f'Augmented {idx+1}', fontweight='bold')
    axes[pos].axis('off')

plt.tight_layout()
plt.show()

print('✅ Augmentation creates diverse variations while preserving document structure!')


## 4. Data Generators

**What are Data Generators?**

Instead of loading all 1,470 training images into memory at once, we use **generators** that:
1. Load images in small batches (e.g., 32 images)
2. Apply preprocessing and augmentation on-the-fly
3. Feed batches to the model during training
4. Repeat for each epoch

**Memory Efficiency:**
- Loading all images: ~1.2 GB RAM
- Using generators: ~50 MB RAM (only one batch at a time)

**Two types of generators:**
1. **Training generator**: WITH augmentation (increases diversity)
2. **Validation/Test generator**: WITHOUT augmentation (evaluate on original images)

**Why no augmentation for validation/test?**
- We want to measure performance on real, unmodified images
- Augmentation is only for training to prevent overfitting


## 4.1. Configure Data Generators

We'll create two `ImageDataGenerator` objects with different configurations.


In [ ]:
# Training generator - WITH augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,              # Normalize to [0,1]
    rotation_range=15,           # Random rotation ±15 degrees
    width_shift_range=0.1,       # Horizontal shift (10% of width)
    height_shift_range=0.1,      # Vertical shift (10% of height)
    horizontal_flip=True,        # Random horizontal flip
    brightness_range=[0.8, 1.2], # Brightness variation (80-120%)
    zoom_range=0.1,              # Random zoom (90-110%)
    fill_mode='nearest'          # Fill empty pixels with nearest value
)

# Validation/Test generator - WITHOUT augmentation
val_test_datagen = ImageDataGenerator(
    rescale=1./255  # Only normalization, no augmentation
)

print('✅ Data generators configured!')
print('\n📊 Configuration summary:')
print('\n🎓 Training Generator:')
print('   • Rescaling: ✓ (to [0,1])')
print('   • Rotation: ✓ (±15°)')
print('   • Shift: ✓ (±10%)')
print('   • Flip: ✓ (horizontal)')
print('   • Brightness: ✓ (80-120%)')
print('   • Zoom: ✓ (90-110%)')

print('\n🔍 Validation/Test Generator:')
print('   • Rescaling: ✓ (to [0,1])')
print('   • Augmentation: ✗ (none)')
print('\n💡 This ensures we evaluate on original, unmodified images')


## 5. Create Data Flow from DataFrames

**Challenge**: Our data is in CSV format (image paths + labels), not in directory structure.

**Solution**: We'll use `flow_from_dataframe()` which:
- Reads image paths from CSV
- Loads images on-the-fly
- Applies augmentation
- Returns batches of (images, labels)

**Parameters:**
- `batch_size=32`: Load 32 images at a time (standard batch size)
- `target_size=(224, 224)`: Resize images
- `class_mode='categorical'`: One-hot encode labels for 3 classes
- `shuffle=True`: Randomize order (important for training)


In [ ]:
# Create generators from dataframes
BATCH_SIZE = 32
TARGET_SIZE = (224, 224)

# Training generator (with augmentation)
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='image_path',
    y_col='document_type',
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

# Validation generator (no augmentation)
val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='image_path',
    y_col='document_type',
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False  # Don't shuffle validation data
)

# Test generator (no augmentation)
test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='image_path',
    y_col='document_type',
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False  # Don't shuffle test data
)

print('\n✅ All generators created successfully!')
print(f'\n📊 Generator details:')
print(f'   Batch size: {BATCH_SIZE}')
print(f'   Target size: {TARGET_SIZE}')
print(f'   Number of classes: {len(train_generator.class_indices)}')
print(f'\n🏷️  Class mapping:')
for class_name, class_idx in train_generator.class_indices.items():
    print(f'   {class_idx}: {class_name}')


## 6. Pipeline Validation

**Critical Step**: Before training, we must verify that our pipeline works correctly!

We'll test:
1. ✓ Can we load a batch successfully?
2. ✓ Are image shapes correct? (32, 224, 224, 3)
3. ✓ Are labels correct? (32, 3) - one-hot encoded
4. ✓ Are pixel values normalized? [0, 1]
5. ✓ Do augmented images look reasonable?

**Why is this important?**
- Bugs in the pipeline will waste hours of training time
- Better to catch issues now than after training fails


In [ ]:
# Load one batch from training generator
batch_images, batch_labels = next(train_generator)

print('🔍 PIPELINE VALIDATION')
print('=' * 60)

# Check shapes
print('\n1. Batch Shapes:')
print(f'   Images: {batch_images.shape}')
print(f'   Expected: (32, 224, 224, 3) ✓' if batch_images.shape == (32, 224, 224, 3) else '   ❌ Wrong shape!')
print(f'   Labels: {batch_labels.shape}')
print(f'   Expected: (32, 3) ✓' if batch_labels.shape == (32, 3) else '   ❌ Wrong shape!')

# Check value ranges
print('\n2. Value Ranges:')
print(f'   Image values: [{batch_images.min():.3f}, {batch_images.max():.3f}]')
print(f'   Expected: [0.000, 1.000] ✓' if 0 <= batch_images.min() and batch_images.max() <= 1 else '   ❌ Wrong range!')
print(f'   Label values: {np.unique(batch_labels)}')

# Check label distribution in batch
print('\n3. Label Distribution in Batch:')
label_indices = np.argmax(batch_labels, axis=1)
class_names = list(train_generator.class_indices.keys())
for idx, class_name in enumerate(class_names):
    count = np.sum(label_indices == idx)
    print(f'   {class_name:15s}: {count} images')

print('\n' + '=' * 60)
print('✅ Pipeline validation passed - ready for training!')


## 6.1. Visualize Batch Samples

Let's display 6 images from the batch to verify that augmentation is working correctly.


In [ ]:
# Display 6 images from the batch
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Sample Batch from Training Generator (With Augmentation)', 
             fontsize=14, fontweight='bold')

for i in range(6):
    row = i // 3
    col = i % 3
    
    # Get image and label
    img = batch_images[i]
    label_idx = np.argmax(batch_labels[i])
    label_name = class_names[label_idx]
    
    # Display
    axes[row, col].imshow(img)
    axes[row, col].set_title(f'{label_name}', fontsize=11, fontweight='bold')
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

print('✅ Images from batch displayed successfully!')
print('💡 Notice: Each image may have slight variations due to augmentation')


## 7. Summary and Key Takeaways

**What we accomplished in this notebook:**

✅ **Image Preprocessing Pipeline**
- Created function to load, resize, and normalize images
- Standardized all images to 224×224 pixels
- Normalized pixel values to [0, 1] range

✅ **Data Augmentation Strategy**
- Configured augmentation for training (rotation, flip, brightness, zoom, shift)
- Kept validation/test data unaugmented for fair evaluation
- Visualized augmentation effects to verify they're realistic

✅ **Data Generators**
- Created efficient generators that load images in batches
- Training generator: WITH augmentation
- Validation/Test generators: WITHOUT augmentation
- Batch size: 32 images

✅ **Pipeline Validation**
- Verified batch shapes: (32, 224, 224, 3) for images
- Verified label shapes: (32, 3) for one-hot encoded labels
- Confirmed value ranges are correct
- Displayed sample batches to ensure quality

**Key Decisions Made:**
1. **Image size**: 224×224 (standard for CNNs and transfer learning)
2. **Normalization**: [0, 1] range (better than [0, 255])
3. **Augmentation**: Moderate and realistic (±15° rotation, not extreme)
4. **Batch size**: 32 (good balance between speed and memory)

**Impact on Training:**
- Original dataset: 1,470 images
- With augmentation: Effectively 10,000+ unique variations
- **~7x increase in training diversity!**

**Ready for Next Step:**
→ **Notebook 03**: Build and train baseline CNN model

The data pipeline is now ready to feed our model during training! 🚀
